In [1]:
# 🚀 Entrenamiento de Modelos RNN para Predicción de Arribos EcoBici
# =======================================================================

# Configuración inicial
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.pytorch
import torch
import yaml
from datetime import datetime
from typing import Dict, Any, List

# Configurar warnings
warnings.filterwarnings('ignore')

# Agregar src al path
sys.path.append('../src')

# Imports de nuestros módulos
from time_series_rnn import EcoBiciTimeSeriesPredictor, BikeStationLSTM
from data_analysis import load_combined_data, filter_data_until_date, temporal_split_data
from train_rnn import (
    create_default_config, 
    run_experiment, 
    load_config, 
    setup_mlflow,
    load_and_prepare_data,
    create_predictor_from_config,
    train_and_evaluate_model,
    log_results_to_mlflow,
    create_visualization_plots
)

# Configurar estilo de plots
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

print(f"Dispositivo PyTorch: {'CUDA disponible' if torch.cuda.is_available() else 'CPU solamente'}")
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")


Dispositivo PyTorch: CPU solamente
MLflow tracking URI: file:///Users/juanfra/Documents/Facultad/3er%20an%CC%83o%20-%201er%20Cuatri/Aprendizaje%20Automatico/TPs/EcoBici-AI/notebooks/mlruns


In [2]:
# 📊 1. Verificación de Datos y Configuración
# ==========================================

data_dir = Path('../data/raw/combined')

# Configurar MLflow con nombres descriptivos
mlflow.set_tracking_uri('../mlruns')
mlflow.set_experiment("EcoBici_RNN_Experiments")

# Cargar configuraciones
config_path = Path('../configs/experiment_configs.yaml')

if config_path.exists():
    with open(config_path, 'r') as f:
        all_configs = yaml.safe_load(f)
    
    print(f"Configuraciones disponibles: {list(all_configs.keys())}")
    
    # Mostrar resumen de cada configuración
    for config_name, config in all_configs.items():
        print(f"\n{config_name}:")
        print(f"   • ΔT: {config['model']['delta_t']} min")
        print(f"   • Secuencia: {config['model']['sequence_length']} pasos")
        print(f"   • Hidden size: {config['model']['hidden_size']}")
        print(f"   • Capas: {config['model']['num_layers']}")
        print(f"   • Épocas: {config['training']['epochs']}")
        print(f"   • Batch size: {config['training']['batch_size']}")
        
    # Actualizar directorio de datos en todas las configuraciones
    for config_name in all_configs:
        all_configs[config_name]['data']['data_dir'] = str(data_dir)
        
else:
    print("Archivo de configuración no encontrado. Usando configuración por defecto.")
    all_configs = {'default': create_default_config()}
    all_configs['default']['data']['data_dir'] = str(data_dir)


Configuraciones disponibles: ['base', 'small_fast', 'large_deep', 'short_window', 'long_window', 'experimental', 'production', 'debug', 'memory_optimized']

base:
   • ΔT: 30 min
   • Secuencia: 12 pasos
   • Hidden size: 128
   • Capas: 2
   • Épocas: 50
   • Batch size: 32

small_fast:
   • ΔT: 30 min
   • Secuencia: 6 pasos
   • Hidden size: 64
   • Capas: 1
   • Épocas: 20
   • Batch size: 64

large_deep:
   • ΔT: 30 min
   • Secuencia: 24 pasos
   • Hidden size: 256
   • Capas: 3
   • Épocas: 100
   • Batch size: 16

short_window:
   • ΔT: 15 min
   • Secuencia: 24 pasos
   • Hidden size: 128
   • Capas: 2
   • Épocas: 60
   • Batch size: 32

long_window:
   • ΔT: 60 min
   • Secuencia: 24 pasos
   • Hidden size: 128
   • Capas: 2
   • Épocas: 40
   • Batch size: 32

experimental:
   • ΔT: 30 min
   • Secuencia: 18 pasos
   • Hidden size: 192
   • Capas: 3
   • Épocas: 75
   • Batch size: 24

production:
   • ΔT: 30 min
   • Secuencia: 12 pasos
   • Hidden size: 128
   • Capas: 2


In [ ]:
# 🎯 2. Entrenamiento de Modelo RNN Individual
# ============================================

# Cache para almacenar datos y resultados
if 'data_cache' not in globals():
    data_cache = {}
    predictor_cache = {}
    results_cache = {}

CONFIG_NAME = 'memory_optimized'  # Cambiar por: 'small_fast', 'large_deep', 'short_window', 'long_window'

print(f"Iniciando entrenamiento con configuración: {CONFIG_NAME}")
print(f"Usando datos de: {data_dir.absolute()}")

custom_config = all_configs[CONFIG_NAME].copy()
custom_config['data']['data_dir'] = str(data_dir)

print(f"🔧 Configuración actualizada:")
print(f"   • Directorio de datos: {custom_config['data']['data_dir']}")
print(f"   • ΔT: {custom_config['model']['delta_t']} minutos")
print(f"   • Épocas: {custom_config['training']['epochs']}")

try:
    experiment_name = custom_config['training']['experiment_name']
    setup_mlflow(experiment_name)
    
    with mlflow.start_run(run_name=f"{CONFIG_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
        # Cargar datos (usando cache)
        if CONFIG_NAME not in data_cache:
            print("📊 Cargando y preparando datos...")
            data_cache[CONFIG_NAME] = load_and_prepare_data(custom_config)
        data_splits = data_cache[CONFIG_NAME]
        
        # Crear predictor (usando cache)
        if CONFIG_NAME not in predictor_cache:
            print("🏗️ Creando predictor...")
            predictor_cache[CONFIG_NAME] = create_predictor_from_config(custom_config)
        predictor = predictor_cache[CONFIG_NAME]
        
        # Entrenar y evaluar (usando cache)
        if CONFIG_NAME not in results_cache:
            print("🎯 Entrenando modelo...")
            results_cache[CONFIG_NAME] = train_and_evaluate_model(predictor, data_splits, custom_config)
        results = results_cache[CONFIG_NAME]
        
        # Log resultados a MLflow
        log_results_to_mlflow(results)
        
        # Crear visualizaciones
        create_visualization_plots(results)
        
        print(f"\n🎉 ¡Experimento {CONFIG_NAME} completado!")
        
        # Mostrar resultados
        eval_results = results['eval_results']
        print(f"\n📊 Resultados de evaluación:")
        
        for split in ['train', 'val', 'test']:
            if split in eval_results:
                metrics = eval_results[split]
                print(f"\n   📈 {split.upper()}:")
                for name, value in metrics.items():
                    print(f"      • {name.upper()}: {value:.4f}")
        
        # Guardar referencia a los resultados
        current_results = results
    
except Exception as e:
    print(f"❌ Error durante el entrenamiento: {str(e)}")
    import traceback
    print("🔍 Traceback completo:")
    traceback.print_exc()
    
    current_results = None


Iniciando entrenamiento con configuración: memory_optimized
Usando datos de: /Users/juanfra/Documents/Facultad/3er año - 1er Cuatri/Aprendizaje Automatico/TPs/EcoBici-AI/notebooks/../data/raw/combined
🔧 Configuración actualizada:
   • Directorio de datos: ../data/raw/combined
   • ΔT: 60 minutos
   • Épocas: 30
🎯 Entrenando modelo...


In [4]:
# 📈 3. Análisis Detallado de Resultados del Modelo
# =================================================

if 'current_results' in locals():
    print("🔍 Analizando resultados del modelo entrenado...")
    
    # Extraer componentes
    predictor = current_results['predictor']
    data = current_results['data']
    training_results = current_results['training_results']
    eval_results = current_results['eval_results']
    
    print(f"\n🏗️ Arquitectura del modelo:")
    print(f"   • Estaciones: {predictor.model.num_stations}")
    print(f"   • Hidden size: {predictor.model.hidden_size}")
    print(f"   • Capas LSTM: {predictor.model.num_layers}")
    print(f"   • Parámetros totales: {sum(p.numel() for p in predictor.model.parameters()):,}")
    
    # Crear visualizaciones adicionales
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f'📊 Análisis Detallado - Configuración: {CONFIG_NAME}', fontsize=16, fontweight='bold')
    
    # 1. Curvas de entrenamiento
    if 'train_losses' in training_results and 'val_losses' in training_results:
        epochs = range(1, len(training_results['train_losses']) + 1)
        axes[0, 0].plot(epochs, training_results['train_losses'], 'b-', label='Training Loss', alpha=0.8)
        axes[0, 0].plot(epochs, training_results['val_losses'], 'r-', label='Validation Loss', alpha=0.8)
        axes[0, 0].set_title('Curvas de Pérdida')
        axes[0, 0].set_xlabel('Época')
        axes[0, 0].set_ylabel('MSE Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # Marcar mejor época
        best_epoch = np.argmin(training_results['val_losses']) + 1
        axes[0, 0].axvline(best_epoch, color='green', linestyle='--', alpha=0.7, 
                          label=f'Mejor época: {best_epoch}')
        axes[0, 0].legend()
    
    # 2. Comparación de métricas por split
    splits = ['train', 'val', 'test']
    metrics = ['mae', 'rmse', 'r2']
    
    metric_data = {metric: [] for metric in metrics}
    available_splits = []
    
    for split in splits:
        if split in eval_results:
            available_splits.append(split)
            for metric in metrics:
                if metric in eval_results[split]:
                    metric_data[metric].append(eval_results[split][metric])
                else:
                    metric_data[metric].append(np.nan)
    
    x_pos = np.arange(len(available_splits))
    width = 0.25
    
    for i, metric in enumerate(metrics):
        if not all(np.isnan(metric_data[metric])):
            axes[0, 1].bar(x_pos + i * width, metric_data[metric], width, 
                          label=metric.upper(), alpha=0.8)
    
    axes[0, 1].set_title('Métricas por Split de Datos')
    axes[0, 1].set_xlabel('Split')
    axes[0, 1].set_ylabel('Valor')
    axes[0, 1].set_xticks(x_pos + width)
    axes[0, 1].set_xticklabels(available_splits)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Distribución de errores (si tenemos predicciones)
    if 'y_pred_val' in data and 'y_val' in data:
        errors = data['y_val'] - data['y_pred_val']
        axes[1, 0].hist(errors.flatten(), bins=50, alpha=0.7, color='skyblue', edgecolor='black')
        axes[1, 0].set_title('Distribución de Errores de Predicción')
        axes[1, 0].set_xlabel('Error (Real - Predicho)')
        axes[1, 0].set_ylabel('Frecuencia')
        axes[1, 0].axvline(0, color='red', linestyle='--', alpha=0.8)
        axes[1, 0].grid(True, alpha=0.3)
        
        # Estadísticas de errores
        error_stats = {
            'Media': np.mean(errors),
            'Mediana': np.median(errors),
            'Std': np.std(errors),
            'MAE': np.mean(np.abs(errors))
        }
        
        stats_text = '\\n'.join([f'{k}: {v:.3f}' for k, v in error_stats.items()])
        axes[1, 0].text(0.05, 0.95, stats_text, transform=axes[1, 0].transAxes, 
                       verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    else:
        axes[1, 0].text(0.5, 0.5, 'Predicciones no disponibles\\nen el set de validación', 
                       ha='center', va='center', transform=axes[1, 0].transAxes)
        axes[1, 0].set_title('Distribución de Errores')
    
    # 4. Performance por estación (top 10)
    if 'station_performance' in eval_results:
        station_perf = eval_results['station_performance']
        top_stations = sorted(station_perf.items(), key=lambda x: x[1]['mae'])[:10]
        
        station_ids = [str(s[0]) for s in top_stations]
        station_maes = [s[1]['mae'] for s in top_stations]
        
        axes[1, 1].barh(range(len(station_ids)), station_maes, alpha=0.7, color='lightgreen')
        axes[1, 1].set_title('Top 10 Estaciones - Menor MAE')
        axes[1, 1].set_xlabel('MAE')
        axes[1, 1].set_ylabel('Estación ID')
        axes[1, 1].set_yticks(range(len(station_ids)))
        axes[1, 1].set_yticklabels(station_ids)
        axes[1, 1].grid(True, alpha=0.3)
    else:
        axes[1, 1].text(0.5, 0.5, 'Performance por estación\\nno disponible', 
                       ha='center', va='center', transform=axes[1, 1].transAxes)
        axes[1, 1].set_title('Performance por Estación')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\\n✅ Análisis completado!")
    
else:
    print("⚠️ No hay resultados disponibles para analizar. Ejecuta primero el entrenamiento.")


🔍 Analizando resultados del modelo entrenado...


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# 🏆 4. Comparación de Múltiples Configuraciones
# ==============================================

# Esta sección permite comparar diferentes configuraciones de manera automática
# Útil para encontrar la mejor arquitectura para el problema

CONFIGS_TO_COMPARE = ['base', 'small_fast', 'large_deep']  # Agregar más según se necesite
QUICK_COMPARISON = True  # Si True, reduce las épocas para comparación rápida

print(f"🔄 Iniciando comparación de configuraciones: {CONFIGS_TO_COMPARE}")
print(f"⚡ Modo rápido: {'Activado' if QUICK_COMPARISON else 'Desactivado'}")

comparison_results = {}

for config_name in CONFIGS_TO_COMPARE:
    print(f"\\n🚀 Entrenando configuración: {config_name}")
    
    try:
        # Cargar configuración
        config = load_config(config_name=config_name)
        
        # Modificar para comparación rápida si está activado
        if QUICK_COMPARISON:
            config['training']['epochs'] = 10  # Reducir épocas para prueba rápida
            config['training']['patience'] = 5
        
        # Ejecutar experimento
        result = run_experiment(
            config_name=config_name,
            save_model=False,  # No guardar modelos en comparación
            create_plots=False  # No crear plots individuales
        )
        
        comparison_results[config_name] = result
        
        # Mostrar resultados rápidos
        eval_res = result['eval_results']
        print(f"   ✅ {config_name} completado:")
        if 'val' in eval_res:
            print(f"      • VAL MAE: {eval_res['val']['mae']:.4f}")
            print(f"      • VAL RMSE: {eval_res['val']['rmse']:.4f}")
            print(f"      • VAL R²: {eval_res['val']['r2']:.4f}")
        
    except Exception as e:
        print(f"   ❌ Error en {config_name}: {str(e)}")
        comparison_results[config_name] = None

print(f"\\n🏁 Comparación completada! Resultados para {len([r for r in comparison_results.values() if r is not None])} configuraciones.")


In [ ]:
# 📊 5. Visualización de Comparación de Modelos
# =============================================

if comparison_results and any(r is not None for r in comparison_results.values()):
    print("📈 Creando visualizaciones comparativas...")
    
    # Extraer métricas para comparación
    configs = []
    val_maes = []
    val_rmses = []
    val_r2s = []
    test_maes = []
    test_rmses = []
    test_r2s = []
    train_times = []
    model_params = []
    
    for config_name, result in comparison_results.items():
        if result is not None:
            configs.append(config_name)
            eval_res = result['eval_results']
            
            # Métricas de validación
            if 'val' in eval_res:
                val_maes.append(eval_res['val']['mae'])
                val_rmses.append(eval_res['val']['rmse'])
                val_r2s.append(eval_res['val']['r2'])
            else:
                val_maes.append(np.nan)
                val_rmses.append(np.nan)
                val_r2s.append(np.nan)
            
            # Métricas de test
            if 'test' in eval_res:
                test_maes.append(eval_res['test']['mae'])
                test_rmses.append(eval_res['test']['rmse'])
                test_r2s.append(eval_res['test']['r2'])
            else:
                test_maes.append(np.nan)
                test_rmses.append(np.nan)
                test_r2s.append(np.nan)
            
            # Información del modelo
            if 'predictor' in result:
                predictor = result['predictor']
                if predictor.model:
                    params = sum(p.numel() for p in predictor.model.parameters())
                    model_params.append(params)
                else:
                    model_params.append(0)
            else:
                model_params.append(0)
            
            # Tiempo de entrenamiento (si está disponible)
            training_res = result.get('training_results', {})
            train_times.append(training_res.get('training_time', 0))
    
    # Crear figura comparativa
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('🏆 Comparación de Configuraciones RNN', fontsize=16, fontweight='bold')
    
    x_pos = np.arange(len(configs))
    
    # 1. MAE Comparison
    axes[0, 0].bar(x_pos - 0.2, val_maes, 0.4, label='Validation', alpha=0.8, color='skyblue')
    axes[0, 0].bar(x_pos + 0.2, test_maes, 0.4, label='Test', alpha=0.8, color='lightcoral')
    axes[0, 0].set_title('Mean Absolute Error (MAE)')
    axes[0, 0].set_ylabel('MAE')
    axes[0, 0].set_xticks(x_pos)
    axes[0, 0].set_xticklabels(configs, rotation=45)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. RMSE Comparison
    axes[0, 1].bar(x_pos - 0.2, val_rmses, 0.4, label='Validation', alpha=0.8, color='lightgreen')
    axes[0, 1].bar(x_pos + 0.2, test_rmses, 0.4, label='Test', alpha=0.8, color='orange')
    axes[0, 1].set_title('Root Mean Square Error (RMSE)')
    axes[0, 1].set_ylabel('RMSE')
    axes[0, 1].set_xticks(x_pos)
    axes[0, 1].set_xticklabels(configs, rotation=45)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. R² Comparison
    axes[0, 2].bar(x_pos - 0.2, val_r2s, 0.4, label='Validation', alpha=0.8, color='gold')
    axes[0, 2].bar(x_pos + 0.2, test_r2s, 0.4, label='Test', alpha=0.8, color='mediumpurple')
    axes[0, 2].set_title('R² Score')
    axes[0, 2].set_ylabel('R²')
    axes[0, 2].set_xticks(x_pos)
    axes[0, 2].set_xticklabels(configs, rotation=45)
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    axes[0, 2].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    
    # 4. Model Parameters
    axes[1, 0].bar(x_pos, model_params, alpha=0.8, color='lightblue')
    axes[1, 0].set_title('Parámetros del Modelo')
    axes[1, 0].set_ylabel('Número de Parámetros')
    axes[1, 0].set_xticks(x_pos)
    axes[1, 0].set_xticklabels(configs, rotation=45)
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Training Time (si está disponible)
    if any(t > 0 for t in train_times):
        axes[1, 1].bar(x_pos, train_times, alpha=0.8, color='lightcyan')
        axes[1, 1].set_title('Tiempo de Entrenamiento')
        axes[1, 1].set_ylabel('Segundos')
        axes[1, 1].set_xticks(x_pos)
        axes[1, 1].set_xticklabels(configs, rotation=45)
        axes[1, 1].grid(True, alpha=0.3)
    else:
        axes[1, 1].text(0.5, 0.5, 'Tiempo de entrenamiento\\nno disponible', 
                       ha='center', va='center', transform=axes[1, 1].transAxes)
        axes[1, 1].set_title('Tiempo de Entrenamiento')
    
    # 6. Efficiency Plot (Performance vs Parameters)
    valid_indices = [i for i, (mae, params) in enumerate(zip(val_maes, model_params)) 
                    if not np.isnan(mae) and params > 0]
    
    if valid_indices:
        scatter_maes = [val_maes[i] for i in valid_indices]
        scatter_params = [model_params[i] for i in valid_indices]
        scatter_configs = [configs[i] for i in valid_indices]
        
        axes[1, 2].scatter(scatter_params, scatter_maes, alpha=0.8, s=100, color='red')
        for i, config in enumerate(scatter_configs):
            axes[1, 2].annotate(config, (scatter_params[i], scatter_maes[i]), 
                              xytext=(5, 5), textcoords='offset points', fontsize=10)
        
        axes[1, 2].set_title('Eficiencia: Performance vs Complejidad')
        axes[1, 2].set_xlabel('Parámetros del Modelo')
        axes[1, 2].set_ylabel('Validation MAE')
        axes[1, 2].grid(True, alpha=0.3)
    else:
        axes[1, 2].text(0.5, 0.5, 'Datos insuficientes\\npara gráfico de eficiencia', 
                       ha='center', va='center', transform=axes[1, 2].transAxes)
        axes[1, 2].set_title('Eficiencia del Modelo')
    
    plt.tight_layout()
    plt.show()
    
    # Crear tabla resumen
    print(f"\\n📋 Tabla Resumen de Comparación:")
    print("=" * 80)
    
    summary_data = {
        'Configuración': configs,
        'Val_MAE': [f"{mae:.4f}" if not np.isnan(mae) else "N/A" for mae in val_maes],
        'Test_MAE': [f"{mae:.4f}" if not np.isnan(mae) else "N/A" for mae in test_maes],
        'Val_R²': [f"{r2:.4f}" if not np.isnan(r2) else "N/A" for r2 in val_r2s],
        'Parámetros': [f"{params:,}" if params > 0 else "N/A" for params in model_params]
    }
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    
    # Identificar mejor modelo
    valid_val_maes = [(i, mae) for i, mae in enumerate(val_maes) if not np.isnan(mae)]
    if valid_val_maes:
        best_idx, best_mae = min(valid_val_maes, key=lambda x: x[1])
        best_config = configs[best_idx]
        print(f"\\n🏆 Mejor configuración: {best_config} (VAL MAE: {best_mae:.4f})")
    
else:
    print("⚠️ No hay resultados de comparación disponibles.")


In [ ]:
# 🔮 6. Predicciones Futuras y Análisis de Casos
# ==============================================

# Esta sección demuestra cómo usar el modelo entrenado para hacer predicciones futuras
# y analiza casos específicos de estaciones

if 'current_results' in locals() and current_results is not None:
    print("🔮 Generando predicciones de ejemplo...")
    
    predictor = current_results['predictor']
    data = current_results['data']
    
    # Ejemplo de predicción futura
    print(f"\\n📈 Ejemplo de Predicción Futura:")
    print(f"   Usando las últimas {predictor.sequence_length} ventanas temporales para predecir la siguiente...")
    
    if 'X_test' in data and len(data['X_test']) > 0:
        # Tomar una secuencia de ejemplo del conjunto de test
        example_sequence = data['X_test'][-1:].copy()  # Última secuencia
        
        print(f"   📊 Dimensiones de la secuencia: {example_sequence.shape}")
        
        # Hacer predicción
        predictor.model.eval()
        with torch.no_grad():
            sequence_tensor = torch.FloatTensor(example_sequence).to(predictor.device)
            prediction = predictor.model(sequence_tensor)
            
            # Desnormalizar si es necesario
            if hasattr(predictor, 'scaler_targets'):
                # Crear array con forma correcta para el scaler
                pred_reshaped = prediction.cpu().numpy().reshape(-1, 1)
                pred_denorm = predictor.scaler_targets.inverse_transform(pred_reshaped)
                prediction_values = pred_denorm.flatten()
            else:
                prediction_values = prediction.cpu().numpy().flatten()
        
        print(f"   🎯 Predicción realizada para {len(prediction_values)} estaciones")
        
        # Mostrar estadísticas de la predicción
        print(f"   📊 Estadísticas de predicción:")
        print(f"      • Arribos totales predichos: {np.sum(prediction_values):.1f}")
        print(f"      • Promedio por estación: {np.mean(prediction_values):.2f}")
        print(f"      • Máximo: {np.max(prediction_values):.2f}")
        print(f"      • Mínimo: {np.min(prediction_values):.2f}")
        
        # Encontrar estaciones con mayor actividad predicha
        if hasattr(predictor, 'station_encoder'):
            # Si tenemos información de estaciones
            top_indices = np.argsort(prediction_values)[-10:][::-1]
            print(f"\\n   🏆 Top 10 estaciones con mayor actividad predicha:")
            for i, idx in enumerate(top_indices, 1):
                print(f"      {i:2d}. Estación {idx}: {prediction_values[idx]:.2f} arribos")
        
        # Visualizar predicción
        plt.figure(figsize=(15, 5))
        
        # Subplot 1: Distribución de predicciones
        plt.subplot(1, 3, 1)
        plt.hist(prediction_values, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
        plt.title('Distribución de Arribos Predichos')
        plt.xlabel('Arribos Predichos')
        plt.ylabel('Número de Estaciones')
        plt.grid(True, alpha=0.3)
        
        # Subplot 2: Top estaciones
        if len(prediction_values) >= 10:
            top_10_values = prediction_values[top_indices[:10]]
            plt.subplot(1, 3, 2)
            plt.bar(range(10), top_10_values, alpha=0.7, color='lightcoral')
            plt.title('Top 10 Estaciones - Arribos Predichos')
            plt.xlabel('Ranking')
            plt.ylabel('Arribos Predichos')
            plt.xticks(range(10), [f"#{i+1}" for i in range(10)])
            plt.grid(True, alpha=0.3)
        
        # Subplot 3: Comparación con valores reales si están disponibles
        if 'y_test' in data and len(data['y_test']) > 0:
            real_values = data['y_test'][-1].flatten()
            
            # Calcular correlación
            correlation = np.corrcoef(prediction_values, real_values)[0, 1]
            
            plt.subplot(1, 3, 3)
            plt.scatter(real_values, prediction_values, alpha=0.6, color='green')
            plt.plot([0, max(real_values)], [0, max(real_values)], 'r--', linewidth=2)
            plt.title(f'Predicho vs Real\\n(Correlación: {correlation:.3f})')
            plt.xlabel('Arribos Reales')
            plt.ylabel('Arribos Predichos')
            plt.grid(True, alpha=0.3)
        else:
            plt.subplot(1, 3, 3)
            plt.text(0.5, 0.5, 'Valores reales\\nno disponibles', 
                    ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('Predicho vs Real')
        
        plt.tight_layout()
        plt.show()
        
    else:
        print("   ⚠️ No hay datos de test disponibles para generar predicciones de ejemplo.")

else:
    print("⚠️ No hay modelo entrenado disponible para hacer predicciones.")
    print("   Ejecuta primero el entrenamiento en las celdas anteriores.")


In [ ]:
# 🧪 7. Experimentos Preparados para Futuras Arquitecturas
# ========================================================

# Esta sección prepara el framework para comparar RNN con otras arquitecturas
# como XGBoost, LightGBM, etc. usando MLflow

print("🧪 Preparando framework para comparar múltiples algoritmos...")

def prepare_for_other_algorithms():
    """
    Función placeholder para preparar datos para otros algoritmos como XGBoost.
    Esta función servirá de base para comparaciones futuras.
    """
    print("📊 Preparando datos para algoritmos no-RNN...")
    
    # Configuración para diferentes tipos de algoritmos
    algorithm_configs = {
        'xgboost': {
            'experiment_name': 'EcoBici_XGBoost_Experiments',
            'model_type': 'tree_based',
            'feature_engineering': True,
            'temporal_features': True
        },
        'lightgbm': {
            'experiment_name': 'EcoBici_LightGBM_Experiments', 
            'model_type': 'tree_based',
            'feature_engineering': True,
            'temporal_features': True
        },
        'catboost': {
            'experiment_name': 'EcoBici_CatBoost_Experiments',
            'model_type': 'tree_based',
            'feature_engineering': False,  # CatBoost maneja features categóricas automáticamente
            'temporal_features': True
        },
        'linear_models': {
            'experiment_name': 'EcoBici_Linear_Experiments',
            'model_type': 'linear',
            'feature_engineering': True,
            'temporal_features': True
        }
    }
    
    return algorithm_configs

# Configurar experimentos para diferentes algoritmos
algorithm_configs = prepare_for_other_algorithms()

print("✅ Framework preparado para comparaciones multi-algoritmo!")
print(f"📋 Algoritmos configurados: {list(algorithm_configs.keys())}")

# Función para registrar baseline de comparación
def log_rnn_baseline_to_mlflow():
    """
    Registra el mejor resultado RNN como baseline para comparaciones futuras.
    """
    if 'current_results' in locals() and current_results is not None:
        eval_results = current_results['eval_results']
        
        # Crear experimento de baselines si no existe
        mlflow.set_experiment("EcoBici_Algorithm_Comparison")
        
        with mlflow.start_run(run_name=f"RNN_Baseline_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
            # Log información del modelo
            mlflow.log_param("algorithm", "RNN_LSTM")
            mlflow.log_param("config_name", CONFIG_NAME if 'CONFIG_NAME' in locals() else "unknown")
            
            # Log métricas de baseline
            if 'val' in eval_results:
                for metric, value in eval_results['val'].items():
                    mlflow.log_metric(f"baseline_val_{metric}", value)
            
            if 'test' in eval_results:
                for metric, value in eval_results['test'].items():
                    mlflow.log_metric(f"baseline_test_{metric}", value)
            
            # Log información adicional
            predictor = current_results['predictor']
            mlflow.log_param("model_parameters", sum(p.numel() for p in predictor.model.parameters()))
            mlflow.log_param("sequence_length", predictor.sequence_length)
            mlflow.log_param("delta_t", predictor.delta_t)
            
            print("✅ Baseline RNN registrado en MLflow para comparaciones futuras")
    else:
        print("⚠️ No hay resultados RNN disponibles para registrar como baseline")

# Registrar baseline si hay resultados disponibles
if 'current_results' in locals():
    log_rnn_baseline_to_mlflow()

print(f"\\n🎯 Próximos pasos recomendados:")
print("1. 🔄 Implementar entrenamiento de XGBoost usando las mismas divisiones de datos")
print("2. 📊 Comparar métricas entre RNN y algoritmos tree-based")  
print("3. ⚡ Evaluar tiempos de entrenamiento e inferencia")
print("4. 🎨 Crear dashboard de comparación en MLflow UI")
print("5. 🔬 Analizar fortalezas/debilidades de cada algoritmo por tipo de estación")

print(f"\\n📱 Para ver experimentos en MLflow UI:")
print(f"   Ejecutar: mlflow ui")
print(f"   Abrir: http://localhost:5000")
